In [8]:
"""
Seq2Seq Nepali Spelling Correction → ONNX Export
Exports both end-to-end model (with greedy decode loop) and separate encoder/decoder variants.
Input: text (misspelled word)
Output: text (corrected word)
"""

import json
import torch
import torch.nn as nn
import numpy as np
from typing import Tuple, Dict, Any, Optional
import onnxruntime as ort


# ============================================================================
# TODO: ADJUST THESE PATHS & HYPERPARAMETERS FOR YOUR MODEL
# ============================================================================
MODEL_CHECKPOINT_PATH = "seq2seq_best.pth"  # e.g., "nepali_correction_model.pth"
TOKENIZER_PATH = "seq2seq_char_tokenizer.json"  # e.g., "tokenizer.json"

# Model hyperparameters (adjust to match your model)
D_MODEL = 64
HIDDEN_SIZE = 64
NUM_LAYERS = 2  # Decoder LSTM layers
VOCAB_SIZE = 82
MAX_LEN = 100
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

# Output paths
ONNX_FULL_PATH = "correction_full.onnx"  # End-to-end with greedy decode
ONNX_ENCODER_PATH = "correction_encoder.onnx"
ONNX_DECODER_PATH = "correction_decoder.onnx"

# ============================================================================
# TOKENIZER UTILS
# ============================================================================

class CharTokenizer:
    """Minimal char-level tokenizer loader."""
    
    def __init__(self, tokenizer_path: str):
        with open(tokenizer_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Expected keys: char2idx, idx2char, special_tokens or similar
        if isinstance(data.get('char2idx'), dict):
            self.char2idx = data['char2idx']
            self.idx2char = {int(k): v for k, v in data['idx2char'].items()}
        else:
            raise ValueError(f"Tokenizer JSON must have 'char2idx' and 'idx2char'. Got keys: {data.keys()}")
        
        self.vocab_size = len(self.char2idx)
        self.pad_idx = data.get('special_tokens', {}).get('PAD', 0)
        self.sos_idx = data.get('special_tokens', {}).get('SOS', 1)
        self.eos_idx = data.get('special_tokens', {}).get('EOS', 2)
        self.unk_idx = data.get('special_tokens', {}).get('UNK', 3)
    
    def encode(self, text: str) -> list:
        """Text → list of char IDs."""
        return [self.char2idx.get(c, self.unk_idx) for c in text]
    
    def decode(self, ids: list) -> str:
        """List of char IDs → text, stopping at EOS."""
        chars = []
        for idx in ids:
            if idx == self.eos_idx:
                break
            if idx != self.pad_idx:
                chars.append(self.idx2char.get(int(idx), '<UNK>'))
        return ''.join(chars)


# ============================================================================
# PLACEHOLDER MODEL CLASSES (YOU REPLACE THESE WITH YOUR ACTUAL MODELS)
# ============================================================================

class Encoder(nn.Module):
    """Placeholder encoder."""
    def __init__(self, vocab_size: int, d_model: int, hidden_size: int, num_layers: int = 1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.lstm = nn.LSTM(
            d_model, hidden_size, num_layers=num_layers,
            batch_first=True, bidirectional=False
        )
        # If you need to compute h, c from enc_out, uncomment below:
        # self.fc_h = nn.Linear(hidden_size, hidden_size)
        # self.fc_c = nn.Linear(hidden_size, hidden_size)
    
    def forward(self, src: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            src: [batch, seq_len]
        Returns:
            enc_out: [batch, seq_len, hidden_size]
            h: [batch, num_layers, hidden_size]  (or [num_layers, batch, hidden_size])
            c: [batch, num_layers, hidden_size]  (or [num_layers, batch, hidden_size])
        """
        embedded = self.embedding(src)  # [batch, seq_len, d_model]
        enc_out, (h, c) = self.lstm(embedded)  # h, c: [num_layers, batch, hidden_size]
        return enc_out, h, c


class Decoder(nn.Module):
    """Placeholder decoder."""
    def __init__(self, vocab_size: int, d_model: int, hidden_size: int, num_layers: int = 1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.lstm = nn.LSTM(
            d_model, hidden_size, num_layers=num_layers,
            batch_first=True
        )
        self.attention_linear = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward_step(
        self,
        last: torch.Tensor,  # [batch, 1]
        h: torch.Tensor,  # [num_layers, batch, hidden_size]
        c: torch.Tensor,  # [num_layers, batch, hidden_size]
        enc_out: torch.Tensor  # [batch, src_len, hidden_size]
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Single decoder step.
        Returns:
            logits: [batch, vocab_size]
            new_h: [num_layers, batch, hidden_size]
            new_c: [num_layers, batch, hidden_size]
        """
        embedded = self.embedding(last)  # [batch, 1, d_model]
        lstm_out, (new_h, new_c) = self.lstm(embedded, (h, c))  # [batch, 1, hidden_size]
        # Dummy attention (replace with your actual attention)
        logits = self.fc(lstm_out[:, 0, :])  # [batch, vocab_size]
        return logits, new_h, new_c


class Seq2SeqCorrector(nn.Module):
    """
    Placeholder seq2seq model.
    TODO: Replace Encoder/Decoder with your actual implementations.
    """
    def __init__(self, vocab_size: int, d_model: int, hidden_size: int, num_layers: int = 1):
        super().__init__()
        self.encoder = Encoder(vocab_size, d_model, hidden_size, num_layers)
        self.decoder = Decoder(vocab_size, d_model, hidden_size, num_layers)
        self.vocab_size = vocab_size
        self.eos_idx = 2
        self.sos_idx = 1
        self.pad_idx = 0
    
    def forward(self, src: torch.Tensor, max_len: int = 100) -> torch.Tensor:
        """
        Full seq2seq forward with greedy decoding.
        Args:
            src: [batch, src_len] character IDs
            max_len: max length of output sequence
        Returns:
            out_ids: [batch, max_len] output character IDs (greedy decoded)
        """
        batch_size = src.size(0)
        device = src.device
        
        # Encode
        enc_out, h, c = self.encoder(src)
        
        # Greedy decode
        out_ids = torch.full((batch_size, max_len), self.pad_idx, dtype=torch.long, device=device)
        last = torch.full((batch_size, 1), self.sos_idx, dtype=torch.long, device=device)
        
        for t in range(max_len):
            logits, h, c = self.decoder.forward_step(last, h, c, enc_out)
            pred_id = logits.argmax(dim=1, keepdim=True)  # [batch, 1]
            out_ids[:, t] = pred_id.squeeze(1)
            
            # Early stop on EOS
            last = pred_id
            # Note: Early stopping with mask is tricky in ONNX; for now we let it run
        
        return out_ids


# ============================================================================
# ONNX EXPORT FUNCTIONS
# ============================================================================

def export_full_model(
    model: nn.Module,
    tokenizer: CharTokenizer,
    onnx_path: str,
    max_len: int = MAX_LEN,
    vocab_size: int = VOCAB_SIZE,
):
    """
    Export seq2seq model with greedy decode loop baked into ONNX graph.
    Uses torch.jit.script to capture the loop.
    """
    print(f"\n{'='*70}")
    print(f"Exporting FULL seq2seq model (with greedy decode loop) → {onnx_path}")
    print(f"{'='*70}")
    
    model.eval()
    
    # Dummy input
    batch_size = 1
    max_src_len = 30
    dummy_src = torch.randint(0, vocab_size, (batch_size, max_src_len), dtype=torch.long)
    
    # Export
    try:
        torch.onnx.export(
            model,
            (dummy_src, max_len),
            onnx_path,
            input_names=['src', 'max_len'],
            output_names=['out_ids'],
            dynamic_axes={
                'src': {0: 'batch_size', 1: 'src_len'},
                'out_ids': {0: 'batch_size', 1: 'max_len'}
            },
            opset_version=17,
            verbose=False,
        )
        print(f"✓ Full model exported to {onnx_path}")
        return True
    except Exception as e:
        print(f"⚠ Full model export failed: {e}")
        print(f"  Falling back to separate encoder/decoder export.")
        return False


def export_encoder(
    encoder: nn.Module,
    onnx_path: str,
    vocab_size: int = VOCAB_SIZE,
):
    """Export encoder only."""
    print(f"\n{'='*70}")
    print(f"Exporting ENCODER → {onnx_path}")
    print(f"{'='*70}")
    
    encoder.eval()
    
    batch_size = 1
    max_src_len = 30
    dummy_src = torch.randint(0, vocab_size, (batch_size, max_src_len), dtype=torch.long)
    
    try:
        torch.onnx.export(
            encoder,
            (dummy_src,),
            onnx_path,
            input_names=['src'],
            output_names=['enc_out', 'h', 'c'],
            dynamic_axes={
                'src': {0: 'batch_size', 1: 'src_len'},
                'enc_out': {0: 'batch_size', 1: 'src_len'},
                'h': {1: 'batch_size'},
                'c': {1: 'batch_size'},
            },
            opset_version=17,
            verbose=False,
        )
        print(f"✓ Encoder exported to {onnx_path}")
        return True
    except Exception as e:
        print(f"✗ Encoder export failed: {e}")
        return False


def export_decoder(
    decoder: nn.Module,
    onnx_path: str,
    hidden_size: int = HIDDEN_SIZE,
    vocab_size: int = VOCAB_SIZE,
):
    """Export decoder (forward_step) only."""
    print(f"\n{'='*70}")
    print(f"Exporting DECODER (forward_step) → {onnx_path}")
    print(f"{'='*70}")
    
    decoder.eval()
    
    # Dummy inputs for forward_step(last, h, c, enc_out)
    batch_size = 1
    num_layers = 2
    src_len = 30
    
    last = torch.tensor([[1]], dtype=torch.long)  # [batch, 1] with SOS
    h = torch.randn(num_layers, batch_size, hidden_size)
    c = torch.randn(num_layers, batch_size, hidden_size)
    enc_out = torch.randn(batch_size, src_len, hidden_size)
    
    try:
        torch.onnx.export(
            decoder.forward_step,
            (last, h, c, enc_out),
            onnx_path,
            input_names=['last', 'h', 'c', 'enc_out'],
            output_names=['logits', 'new_h', 'new_c'],
            dynamic_axes={
                'last': {0: 'batch_size'},
                'h': {1: 'batch_size'},
                'c': {1: 'batch_size'},
                'enc_out': {0: 'batch_size', 1: 'src_len'},
                'logits': {0: 'batch_size'},
                'new_h': {1: 'batch_size'},
                'new_c': {1: 'batch_size'},
            },
            opset_version=17,
            verbose=False,
        )
        print(f"✓ Decoder exported to {onnx_path}")
        return True
    except Exception as e:
        print(f"✗ Decoder export failed: {e}")
        return False


# ============================================================================
# INFERENCE & SANITY CHECK
# ============================================================================

def correct_text_pytorch(
    text: str,
    model: nn.Module,
    tokenizer: CharTokenizer,
    device: str = 'cpu',
    max_len: int = MAX_LEN,
) -> str:
    """Run inference in PyTorch."""
    model.eval()
    with torch.no_grad():
        # Encode input
        src_ids = tokenizer.encode(text)
        src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)
        
        # Forward
        out_ids = model(src_tensor, max_len=max_len)
        
        # Decode output
        out_ids_list = out_ids[0].cpu().numpy().tolist()
        corrected = tokenizer.decode(out_ids_list)
    
    return corrected


def correct_text_onnx(
    text: str,
    onnx_path: str,
    tokenizer: CharTokenizer,
    max_len: int = MAX_LEN,
) -> str:
    """Run inference in ONNX Runtime."""
    try:
        sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    except Exception as e:
        print(f"✗ Failed to load ONNX model: {e}")
        return None
    
    # Encode input
    src_ids = tokenizer.encode(text)
    src_tensor = np.array([src_ids], dtype=np.int64)
    
    # Forward
    try:
        outputs = sess.run(None, {'src': src_tensor})
        out_ids = outputs[0][0]  # [max_len]
    except Exception as e:
        print(f"✗ ONNX inference failed: {e}")
        return None
    
    # Decode output
    corrected = tokenizer.decode(out_ids.tolist())
    return corrected


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("\n" + "="*70)
    print("Seq2Seq Nepali Spelling Correction → ONNX Export")
    print("="*70)
    
    # Load tokenizer
    print(f"\nLoading tokenizer from: {TOKENIZER_PATH}")
    try:
        tokenizer = CharTokenizer(TOKENIZER_PATH)
        print(f"✓ Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
    except FileNotFoundError:
        print(f"✗ Tokenizer not found at {TOKENIZER_PATH}")
        print("   Create a sample tokenizer? (y/n): ", end="")
        if input().lower() == 'y':
            create_sample_tokenizer(TOKENIZER_PATH)
            tokenizer = CharTokenizer(TOKENIZER_PATH)
            print(f"✓ Sample tokenizer created and loaded.")
        else:
            return
    
    # Load model checkpoint
    print(f"\nLoading model checkpoint from: {MODEL_CHECKPOINT_PATH}")
    try:
        model = Seq2SeqCorrector(
            vocab_size=VOCAB_SIZE,
            d_model=D_MODEL,
            hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS,
        )
        checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location='cpu')
        
        # Handle both state_dict and full checkpoint
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
        else:
            model.load_state_dict(checkpoint)
        
        print(f"✓ Model loaded and checkpoint applied.")
    except FileNotFoundError:
        print(f"✗ Checkpoint not found at {MODEL_CHECKPOINT_PATH}")
        print("   Using untrained model for demo. Continue? (y/n): ", end="")
        if input().lower() != 'y':
            return
    except Exception as e:
        print(f"✗ Failed to load checkpoint: {e}")
        return
    
    model.eval()
    
    # Export ONNX files
    print("\n" + "="*70)
    print("EXPORTING ONNX FILES")
    print("="*70)
    
    # 1. Full model with greedy decode loop
    full_ok = export_full_model(
        model, tokenizer, ONNX_FULL_PATH, max_len=MAX_LEN, vocab_size=VOCAB_SIZE
    )
    
    # 2. Encoder only (fallback)
    encoder_ok = export_encoder(model.encoder, ONNX_ENCODER_PATH, vocab_size=VOCAB_SIZE)
    
    # 3. Decoder only (fallback)
    decoder_ok = export_decoder(
        model.decoder, ONNX_DECODER_PATH, hidden_size=HIDDEN_SIZE, vocab_size=VOCAB_SIZE
    )
    
    # Sanity checks
    print("\n" + "="*70)
    print("SANITY CHECKS")
    print("="*70)
    
    test_words = ["नेपाल", "स्कुल", "गर्नु"]  # Sample Nepali words
    
    if full_ok:
        print(f"\nTesting full ONNX model ({ONNX_FULL_PATH}):")
        print("-" * 70)
        for word in test_words:
            try:
                pytorch_result = correct_text_pytorch(word, model, tokenizer)
                onnx_result = correct_text_onnx(word, ONNX_FULL_PATH, tokenizer)
                
                print(f"  Input:     {word}")
                print(f"  PyTorch:   {pytorch_result}")
                print(f"  ONNX:      {onnx_result}")
                
                if pytorch_result == onnx_result:
                    print(f"  ✓ Match!")
                else:
                    print(f"  ⚠ Mismatch (may be normal if model is untrained)")
                print()
            except Exception as e:
                print(f"  ✗ Error testing '{word}': {e}\n")
    else:
        print("\n⚠ Full model export failed. Using encoder+decoder fallback for testing.")
    
    print("="*70)
    print("EXPORT COMPLETE")
    print("="*70)
    print(f"\nGenerated files:")
    if full_ok:
        print(f"  ✓ {ONNX_FULL_PATH} (end-to-end with greedy decode)")
    else:
        print(f"  ✗ {ONNX_FULL_PATH} (export failed)")
    print(f"  {'✓' if encoder_ok else '✗'} {ONNX_ENCODER_PATH} (encoder only)")
    print(f"  {'✓' if decoder_ok else '✗'} {ONNX_DECODER_PATH} (decoder only)")
    print("\nNext steps:")
    print(f"  1. Adjust model paths in the TODO section at top of script")
    print(f"  2. Replace placeholder Encoder/Decoder with your actual model classes")
    print(f"  3. Run: python export_full.py")


def create_sample_tokenizer(path: str):
    """Create a minimal sample tokenizer for testing."""
    chars = list("अआइईउऊऋएऐओऔकखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसहा" + "0123456789")
    char2idx = {c: i for i, c in enumerate(chars, start=4)}  # Start from 4 for special tokens
    char2idx['<PAD>'] = 0
    char2idx['<SOS>'] = 1
    char2idx['<EOS>'] = 2
    char2idx['<UNK>'] = 3
    
    idx2char = {i: c for c, i in char2idx.items()}
    
    tokenizer_data = {
        'char2idx': char2idx,
        'idx2char': idx2char,
        'special_tokens': {
            'PAD': 0,
            'SOS': 1,
            'EOS': 2,
            'UNK': 3,
        }
    }
    
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(tokenizer_data, f, ensure_ascii=False, indent=2)
    
    print(f"✓ Sample tokenizer created at {path}")


if __name__ == '__main__':
    main()


Seq2Seq Nepali Spelling Correction → ONNX Export

Loading tokenizer from: seq2seq_char_tokenizer.json
✗ Tokenizer not found at seq2seq_char_tokenizer.json
   Create a sample tokenizer? (y/n): ✓ Sample tokenizer created at seq2seq_char_tokenizer.json
✓ Sample tokenizer created and loaded.

Loading model checkpoint from: seq2seq_best.pth
✗ Failed to load checkpoint: Error(s) in loading state_dict for Seq2SeqCorrector:
	Missing key(s) in state_dict: "encoder.lstm.weight_ih_l0", "encoder.lstm.weight_hh_l0", "encoder.lstm.bias_ih_l0", "encoder.lstm.bias_hh_l0", "encoder.lstm.weight_ih_l1", "encoder.lstm.weight_hh_l1", "encoder.lstm.bias_ih_l1", "encoder.lstm.bias_hh_l1", "decoder.lstm.weight_ih_l0", "decoder.lstm.weight_hh_l0", "decoder.lstm.bias_ih_l0", "decoder.lstm.bias_hh_l0", "decoder.lstm.weight_ih_l1", "decoder.lstm.weight_hh_l1", "decoder.lstm.bias_ih_l1", "decoder.lstm.bias_hh_l1", "decoder.attention_linear.weight", "decoder.attention_linear.bias", "decoder.fc.weight", "decoder.fc